In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [25]:
df = pd.read_csv('training.csv')

In [26]:
df.head(10)

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0
5,TransactionId_23223,BatchId_25954,AccountId_1078,SubscriptionId_4238,CustomerId_1432,UGX,256,ProviderId_6,ProductId_3,airtime,ChannelId_3,2000.0,2000,2018-11-15T03:35:10Z,2,0
6,TransactionId_118063,BatchId_118460,AccountId_2442,SubscriptionId_1980,CustomerId_2858,UGX,256,ProviderId_5,ProductId_3,airtime,ChannelId_3,10000.0,10000,2018-11-15T03:44:31Z,4,0
7,TransactionId_100640,BatchId_38561,AccountId_4841,SubscriptionId_3829,CustomerId_2858,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-500.0,500,2018-11-15T03:45:13Z,2,0
8,TransactionId_51905,BatchId_93774,AccountId_272,SubscriptionId_4731,CustomerId_598,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,500.0,500,2018-11-15T04:14:59Z,2,0
9,TransactionId_130161,BatchId_82409,AccountId_710,SubscriptionId_920,CustomerId_1053,UGX,256,ProviderId_1,ProductId_15,financial_services,ChannelId_3,600.0,600,2018-11-15T04:31:48Z,2,0


# Data standardisation

In [27]:
# data standadisation

df.columns = df.columns.str.lower()

df['transactionstarttime'] = pd.to_datetime(df['transactionstarttime'])
strings = list(df.columns[df.dtypes == 'object'])

for col in strings:
  df[col] = df[col].str.lower().str.replace(' ', '_')



In [28]:
#check for duplicates
df.duplicated().sum()

np.int64(0)

In [29]:
#check for data types
df.dtypes

transactionid                           str
batchid                                 str
accountid                               str
subscriptionid                          str
customerid                              str
currencycode                            str
countrycode                           int64
providerid                              str
productid                               str
productcategory                         str
channelid                               str
amount                              float64
value                                 int64
transactionstarttime    datetime64[us, UTC]
pricingstrategy                       int64
fraudresult                           int64
dtype: object

In [30]:
df.isnull().sum()

transactionid           0
batchid                 0
accountid               0
subscriptionid          0
customerid              0
currencycode            0
countrycode             0
providerid              0
productid               0
productcategory         0
channelid               0
amount                  0
value                   0
transactionstarttime    0
pricingstrategy         0
fraudresult             0
dtype: int64

# Exploratory data analysis
- Mutual information
- correlation coeficients
- feature engineering

In [31]:
#global fraud
global_fraud = df.fraudresult.mean()
global_fraud

np.float64(0.00201752001839811)

In [32]:
# average amount for a frodulant transaction is 1 500 000
df[['value', 'fraudresult']][df.fraudresult == 1].mean()

value          1.561820e+06
fraudresult    1.000000e+00
dtype: float64

In [33]:
df['hour']       = df['transactionstarttime'].dt.hour
df['day']        = df['transactionstarttime'].dt.day
df['month']      = df['transactionstarttime'].dt.month
df['year']       = df['transactionstarttime'].dt.year
df['dayofweek']  = df['transactionstarttime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['is_weekend'] = df['transactionstarttime'].dt.dayofweek >= 5

In [34]:
df = df[['value','fraudresult', 'hour', 'day', 'month', 'year','dayofweek', 'is_weekend']]

In [35]:
df

,value,fraudresult,hour,day,month,year,dayofweek,is_weekend
0,1000,0,2,15,11,2018,3,False
1,20,0,2,15,11,2018,3,False
2,500,0,2,15,11,2018,3,False
3,21800,0,3,15,11,2018,3,False
4,644,0,3,15,11,2018,3,False
...,...,...,...,...,...,...,...,...
95657,1000,0,9,13,2,2019,2,False
95658,1000,0,9,13,2,2019,2,False
95659,20,0,9,13,2,2019,2,False
95660,3000,0,10,13,2,2019,2,False


In [39]:
df.groupby('is_weekend')['fraudresult'].mean()

is_weekend
False    0.002017
True     0.002021
Name: fraudresult, dtype: float64